# SC-GRC Hybrid — Signal Stage 2/4 — Questions 85–168

This notebook is a **single-run fixed shard**. It performs only the new hybrid SLM-side work on 84 questions: KB-only hybrid retrieval, k=3 FT-SLM sampling, self-consistency, groundedness, and code-mixing. It does **not** rerun Base, FT, old RAG, or old cache experiments.

Run this notebook once, save `/kaggle/working/sc_grc_state` as a Kaggle Dataset, then use that Dataset as `INPUT_STATE_DIR` for the next stage.

In [1]:
%%capture
import os, subprocess, sys
cuda_lib = "/usr/local/cuda/lib64"
ld = os.environ.get("LD_LIBRARY_PATH", "")
if cuda_lib not in ld:
    os.environ["LD_LIBRARY_PATH"] = f"{cuda_lib}:{ld}"
subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-cache-dir",
    "torch==2.3.1","torchvision==0.18.1","--extra-index-url","https://download.pytorch.org/whl/cu121"])
subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-cache-dir",
    "pandas==2.2.2","scipy==1.13.1","matplotlib==3.8.4","triton==2.3.1","bitsandbytes==0.44.1",
    "transformers==4.46.3","peft==0.13.2","accelerate==0.34.2","sentencepiece","sacrebleu",
    "rapidfuzz","openpyxl","sentence-transformers","rank_bm25","langchain-text-splitters","faiss-gpu-cu12","FlagEmbedding",
    "scikit-learn","tqdm","bert-score","rouge-score","joblib"])


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 MB 242.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 168.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 264.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 356.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 322.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 334.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 319.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 347.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 329.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 337.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 323.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 252.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.3.1+cu121 which is incompatible.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 12.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 282.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 225.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 259.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 225.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 MB 155.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 264.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 382.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 388.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 329.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 267.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
tsfresh 0.21.1 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.13.1 which is incompatible.
pointpats 2.5.5 requires matplotlib>=3.9, but you have matplotlib 3.8.4 which is incompatible.
access 1.1.10.post3 requires scipy>=1.14.1, but you have scipy 1.13.1 which is incompatible.


In [2]:
# ========================= USER CONFIG =========================
MODEL_ID_SLM = "Qwen/Qwen2.5-3B-Instruct"
SLM_ADAPTER_DIR = "/kaggle/input/datasets/mrnotalent/ewu-qwen-adapter-checkpoint1632"  # EDIT if needed
KB_DIR = "/kaggle/input/datasets/mohuaakter/ewu-dataset-jsonl-pairs/KB/KB"  # KB ONLY; no scraped data
GROUND_TRUTH_XLSX = "/kaggle/input/datasets/mohuaakter/ewu-dataset-jsonl-pairs/University_Chatbot_Questions_1110_GROUND_TRUTHS_VERIFIED.xlsx"  # EDIT
INPUT_STATE_DIR = "/kaggle/input/datasets/bxgdhdgsg/hbrun1/sc_grc_state"  # empty for notebook 01; set to previous notebook's saved Dataset folder thereafter
WORK_STATE_DIR = "/kaggle/working/sc_grc_state"

SEED = 42
TOP_K_DENSE, TOP_K_SPARSE, TOP_K_FINAL = 10, 10, 4
K_SAMPLES = 3
SC_GRC_TEMPERATURE = 0.7
SC_GRC_TOP_P = 0.9
QUALITY_FLOOR_BERTSCORE = 0.85
MAX_NEW_TOKENS = 200

# Fixed shard — do not change.
SHARD_START=84
SHARD_END=168


In [3]:

import os, json, re, random, shutil, time
from collections import Counter
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import PeftModel

os.makedirs(WORK_STATE_DIR, exist_ok=True)
if INPUT_STATE_DIR and os.path.isdir(INPUT_STATE_DIR) and INPUT_STATE_DIR != WORK_STATE_DIR:
    for name in os.listdir(INPUT_STATE_DIR):
        src, dst = os.path.join(INPUT_STATE_DIR,name), os.path.join(WORK_STATE_DIR,name)
        if os.path.isfile(src) and not os.path.exists(dst):
            shutil.copy2(src,dst)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE, "GPUs:", torch.cuda.device_count() if torch.cuda.is_available() else 0)

# Exact 333-item evaluation construction from the completed SLM notebook (§8.1):
# 1 question per Category × Difficulty (111), evaluated in all 3 languages = 333.
LANGUAGE_COLUMNS = {"english":"English Query", "bangla":"Bangla Query", "banglish":"Banglish Query"}
DIFFICULTY_ALIAS = {"Easy":"Simple", "Medium":"Normal", "Hard":"Complex"}
EVAL_CSV = os.path.join(WORK_STATE_DIR, "eval_333.csv")
if not os.path.exists(EVAL_CSV):
    corpus = pd.read_excel(GROUND_TRUTH_XLSX, sheet_name="Question Corpus")
    rows=[]
    for _, r in corpus.iterrows():
        qnum=r["Q#"]
        for language,qcol in LANGUAGE_COLUMNS.items():
            query=r[qcol]
            if pd.isna(query) or not str(query).strip():
                continue
            rows.append({
                "Q#":qnum,"language":language,"query":str(query).strip(),
                "Category":r.get("Category",""),"Difficulty":r.get("Difficulty",""),
                "Difficulty Group":DIFFICULTY_ALIAS.get(str(r.get("Difficulty","")),str(r.get("Difficulty",""))),
                "Group":r.get("Group",""),"Query Style":r.get("Query Style",""),
                "Noise Type":r.get("Noise Type",""),"Intent":r.get("Intent",""),
                "Ground Truth Type":r.get("Ground Truth Type",""),
                "ground_truth":str(r["Ground Truth Answer (Canonical English)"]).strip()
            })
    xlsx_test_df=pd.DataFrame(rows)
    selected_qnums=(corpus.groupby(["Category","Difficulty"],dropna=False)["Q#"]
        .apply(lambda s:s.sample(n=min(1,len(s)),random_state=SEED)).reset_index(drop=True))
    xlsx_eval_df=(xlsx_test_df[xlsx_test_df["Q#"].isin(selected_qnums)]
        .sample(frac=1,random_state=SEED).reset_index(drop=True))
    if len(xlsx_eval_df)!=333:
        raise ValueError(f"Expected exactly 333 evaluation items; got {len(xlsx_eval_df)}")
    xlsx_eval_df.to_csv(EVAL_CSV,index=False,encoding="utf-8-sig")
else:
    xlsx_eval_df=pd.read_csv(EVAL_CSV)

if len(xlsx_eval_df)!=333:
    raise ValueError(f"eval_333.csv must contain 333 rows; found {len(xlsx_eval_df)}")
test_prompts_eval=xlsx_eval_df["query"].tolist()
test_references_eval=xlsx_eval_df["ground_truth"].tolist()
test_langs_eval=xlsx_eval_df["language"].tolist()
print("333-item evaluation set ready")

from langchain_text_splitters import MarkdownTextSplitter
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from FlagEmbedding import FlagReranker

SKIP_KEYS={"navigation_links","page_info","summary"}
def _flatten_record(d):
    return ". ".join(f"{k.replace('_',' ')}: {v}" for k,v in d.items() if not isinstance(v,(dict,list)) and v not in (None,""))
def _walk_json(obj,file_tag,context=""):
    docs=[]
    if isinstance(obj,dict):
        if obj and all(not isinstance(v,(dict,list)) for v in obj.values()):
            t=_flatten_record(obj)
            if t: docs.append((file_tag,f"{context} {t}".strip()))
            return docs
        for k,v in obj.items():
            if k in SKIP_KEYS: continue
            docs.extend(_walk_json(v,file_tag,f"{context} {k}".strip()))
    elif isinstance(obj,list):
        for item in obj: docs.extend(_walk_json(item,file_tag,context))
    return docs

def load_kb_only(kb_dir):
    if not os.path.isdir(kb_dir): raise FileNotFoundError(f"KB_DIR not found: {kb_dir}")
    splitter=MarkdownTextSplitter(chunk_size=500,chunk_overlap=80)
    chunks=[]; sources=[]
    for root,_,files in os.walk(kb_dir):
        for fn in sorted(files):
            p=os.path.join(root,fn); rel=os.path.relpath(p,kb_dir)
            try:
                if fn.lower().endswith(('.md','.markdown','.txt')):
                    text=open(p,encoding='utf-8',errors='ignore').read().strip()
                    if text:
                        for ch in splitter.split_text(text): chunks.append(ch); sources.append(rel)
                elif fn.lower().endswith('.json'):
                    obj=json.load(open(p,encoding='utf-8'))
                    for src,text in _walk_json(obj,rel):
                        if text: chunks.append(text); sources.append(src)
            except Exception as e:
                print('[skip]',p,type(e).__name__,e)
    return chunks,sources

KB_JSON=os.path.join(WORK_STATE_DIR,'kb.json')
FAISS_PATH=os.path.join(WORK_STATE_DIR,'dense_index.faiss')
KB_META=os.path.join(WORK_STATE_DIR,'kb_meta.json')
if os.path.exists(KB_JSON) and os.path.exists(FAISS_PATH) and os.path.exists(KB_META):
    kb_chunks=json.load(open(KB_JSON,encoding='utf-8'))
    kb_sources=json.load(open(KB_META,encoding='utf-8'))['sources']
    dense_index=faiss.read_index(FAISS_PATH)
else:
    kb_chunks,kb_sources=load_kb_only(KB_DIR)
    if not kb_chunks: raise RuntimeError("No KB chunks found")
    e0=SentenceTransformer("BAAI/bge-m3",device=DEVICE); e0.max_seq_length=512
    embs=np.asarray(e0.encode(kb_chunks,batch_size=16,normalize_embeddings=False,show_progress_bar=True),dtype='float32')
    faiss.normalize_L2(embs)
    dense_index=faiss.IndexFlatIP(embs.shape[1]); dense_index.add(embs)
    json.dump(kb_chunks,open(KB_JSON,'w',encoding='utf-8'),ensure_ascii=False)
    json.dump({'sources':kb_sources,'n_chunks':len(kb_chunks)},open(KB_META,'w',encoding='utf-8'),ensure_ascii=False,indent=2)
    faiss.write_index(dense_index,FAISS_PATH)
print('KB-only chunks:',len(kb_chunks))

# Hybrid retrieval stack, KB-only.
embedder=SentenceTransformer("BAAI/bge-m3",device=DEVICE); embedder.max_seq_length=512
reranker=FlagReranker("BAAI/bge-reranker-v2-m3",use_fp16=True,device=DEVICE)
bm25=BM25Okapi([re.findall(r"[\w\u0980-\u09FF]+",c.lower()) for c in kb_chunks])
def embed_texts(texts,batch_size=16):
    return np.asarray(embedder.encode(texts,batch_size=batch_size,normalize_embeddings=False,show_progress_bar=False),dtype='float32')
def hybrid_retrieve(query, top_k=TOP_K_FINAL):
    q = embed_texts([query])
    faiss.normalize_L2(q)

    _, di = dense_index.search(
        q,
        min(TOP_K_DENSE, len(kb_chunks))
    )

    ss = bm25.get_scores(
        re.findall(r"[\w\u0980-\u09FF]+", query.lower())
    )
    si = np.argsort(ss)[::-1][:min(TOP_K_SPARSE, len(kb_chunks))]

    ids = sorted(set(di[0].tolist()) | set(si.tolist()))
    pairs = [[query, kb_chunks[i]] for i in ids]

    raw_scores = reranker.compute_score(pairs, normalize=True)

    # Make the reranker output a 1-D scalar score array.
    rs = np.asarray(raw_scores, dtype=np.float32).reshape(-1)

    if len(rs) != len(ids):
        raise RuntimeError(
            f"Reranker returned {len(rs)} scores for {len(ids)} candidate chunks. "
            f"Raw shape={np.asarray(raw_scores).shape}"
        )

    order = np.argsort(rs)[::-1][:top_k]

    return [
        {
            'source': kb_sources[ids[int(j)]],
            'chunk': kb_chunks[ids[int(j)]],
            'score': float(rs[int(j)])
        }
        for j in order
    ]

SYSTEM_PROMPT=("You are the East West University (EWU) student support assistant. "
               "Answer the student's question directly, accurately, and helpfully. "
               "Do not invent university policies or facts. If the provided context is insufficient, say so clearly.")
def build_rag_prompt(query,retrieved):
    ctx="\n\n".join(f"[{r['source']}]\n{r['chunk']}" for r in retrieved) if retrieved else "(no relevant context retrieved)"
    user=f"Context:\n{ctx}\n\nQuestion: {query}\n\nAnswer using only the context above, following the system rules."
    return f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{user}<|im_end|>\n<|im_start|>assistant\n"

# Load FT-SLM for the new SC-GRC signal stage only.
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_use_double_quant=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.float16)
base_slm=AutoModelForCausalLM.from_pretrained(MODEL_ID_SLM,quantization_config=bnb,device_map="auto",trust_remote_code=True,torch_dtype=torch.float16,attn_implementation="eager")
slm=PeftModel.from_pretrained(base_slm,SLM_ADAPTER_DIR).eval(); slm.config.use_cache=True
st=AutoTokenizer.from_pretrained(SLM_ADAPTER_DIR,trust_remote_code=True,padding_side="right")
st.pad_token=st.pad_token or st.eos_token

@torch.no_grad()
def generate_k_samples(query, retrieved):
    p = build_rag_prompt(query, retrieved)

    # Prevent very long RAG contexts from causing another large VRAM spike.
    enc = st(
        p,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to(slm.device)

    samples = []
    n = enc["input_ids"].shape[1]

    for _ in range(K_SAMPLES):
        torch.cuda.empty_cache()

        out = slm.generate(
            **enc,
            max_new_tokens=128,
            do_sample=True,
            temperature=SC_GRC_TEMPERATURE,
            top_p=SC_GRC_TOP_P,
            num_return_sequences=1,
            pad_token_id=st.pad_token_id,
            use_cache=True,
        )

        samples.append(
            st.decode(
                out[0][n:],
                skip_special_tokens=True
            ).strip()
        )

        del out
        torch.cuda.empty_cache()

    return samples

def self_consistency_score(samples):
    if len(samples)<2:return 1.0
    e=embed_texts(samples); faiss.normalize_L2(e)
    return float(np.mean([float(np.dot(e[i],e[j])) for i in range(len(samples)) for j in range(i+1,len(samples))]))

def pick_representative_sample(samples):
    if len(samples)==1:return samples[0]
    e=embed_texts(samples); faiss.normalize_L2(e); c=e.mean(axis=0,keepdims=True); faiss.normalize_L2(c)
    return samples[int(np.argmax((e@c.T).reshape(-1)))]

# Groundedness uses local NLI; kept local and on GPU if available.
NLI_MODEL_ID="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
nli_tok=AutoTokenizer.from_pretrained(NLI_MODEL_ID)
nli=AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_ID).to(DEVICE).eval()
NLI_ENT=int([k for k,v in nli.config.id2label.items() if v.lower().startswith('entail')][0])
def _split_sentences(t): return [x.strip() for x in re.split(r'(?<=[।.!?])\s+',t.strip()) if x.strip()]
@torch.no_grad()
def groundedness_score(answer,retrieved):
    sents=_split_sentences(answer)
    if not sents or not retrieved:return 0.0
    ctx=' '.join(r['chunk'] for r in retrieved)
    vals=[]
    for s in sents:
        enc=nli_tok([ctx],[s],truncation=True,max_length=512,padding=True,return_tensors='pt').to(DEVICE)
        vals.append(int(nli(**enc).logits.argmax(dim=-1).item())==NLI_ENT)
    return float(np.mean(vals))

def compute_cmi(text):
    ban=re.compile(r'[\u0980-\u09FF]'); lat=re.compile(r'[A-Za-z]+'); words=re.findall(r'[\w\u0980-\u09FF]+',text)
    if not words:return 0.0
    nb=sum(1 for w in words if ban.search(w)); nl=sum(1 for w in words if lat.fullmatch(w)); n=nb+nl
    return 0.0 if n==0 else 100.0*(n-max(nb,nl))/n


DEVICE: cuda GPUs: 2
333-item evaluation set ready
KB-only chunks: 2292


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

In [4]:

# Fixed shard: this notebook is intended to be run ONCE.
START = 84
END = 168
SIGNAL_FILE=os.path.join(WORK_STATE_DIR,'sc_grc_signals.jsonl')
rows={}
if os.path.exists(SIGNAL_FILE):
    with open(SIGNAL_FILE,encoding='utf-8') as f:
        for line in f:
            if line.strip():
                r=json.loads(line); rows[int(r['idx'])]=r
pending=list(range(START,END))
missing=[i for i in pending if i not in rows]
if missing!=pending:
    raise RuntimeError(f"This fixed-run notebook detected existing rows in its shard: {missing[:5]}. Do not rerun this notebook; use the next stage/state.")
print(f"Processing fixed shard {START}:{END} = {END-START} questions. This notebook is intended to be run once.")
for idx in tqdm(pending,desc=f'SC-GRC signals {START}:{END}'):
    q=test_prompts_eval[idx]
    retrieved=hybrid_retrieve(q)
    samples=generate_k_samples(q,retrieved)
    rep=pick_representative_sample(samples)
    sc=self_consistency_score(samples)
    gs=groundedness_score(rep,retrieved)
    cmi=compute_cmi(rep)
    r={'idx':idx,'query':q,'lang':test_langs_eval[idx],'representative_answer':rep,'samples':samples,'self_consistency':sc,'groundedness':gs,'cmi':cmi}
    with open(SIGNAL_FILE,'a',encoding='utf-8') as f: f.write(json.dumps(r,ensure_ascii=False)+'\n')
    rows[idx]=r

# Notebook 04 additionally trains the router after all 333 rows exist.
if END==333:
    from bert_score import score as bert_score_fn
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    import joblib
    all_rows={}
    with open(SIGNAL_FILE,encoding='utf-8') as f:
        for line in f:
            if line.strip():
                r=json.loads(line); all_rows[int(r['idx'])]=r
    if len(all_rows)!=333:
        raise RuntimeError(f"Expected 333 signal rows before router training; found {len(all_rows)}")
    ordered=pd.DataFrame([all_rows[i] for i in range(333)])
    _,_,bert_f1=bert_score_fn(ordered['representative_answer'].tolist(),test_references_eval,model_type='bert-base-multilingual-cased',device=DEVICE,batch_size=8,verbose=False)
    ordered['rep_bertscore_f1']=bert_f1.tolist()
    ordered['needs_escalation']=(ordered['rep_bertscore_f1']<QUALITY_FLOOR_BERTSCORE).astype(int)
    X=ordered[['self_consistency','groundedness','cmi']].copy(); X['inv_consistency']=1-X['self_consistency']; X['inv_groundedness']=1-X['groundedness']; X=X[['inv_consistency','inv_groundedness','cmi']]
    y=ordered['needs_escalation']
    strat=y if y.nunique()>1 else None
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.4,random_state=SEED,stratify=strat)
    clf=LogisticRegression(class_weight='balanced',random_state=SEED,max_iter=1000).fit(Xtr,ytr)
    ordered['escalate_pred']=clf.predict(X); ordered['escalate_proba']=clf.predict_proba(X)[:,1]
    ordered.to_csv(os.path.join(WORK_STATE_DIR,'sc_grc_signals_with_router.csv'),index=False,encoding='utf-8-sig')
    joblib.dump(clf,os.path.join(WORK_STATE_DIR,'sc_grc_router.joblib'))
    json.dump({'quality_floor_bertscore':QUALITY_FLOOR_BERTSCORE,'k_samples':K_SAMPLES,'temperature':SC_GRC_TEMPERATURE,'top_p':SC_GRC_TOP_P,'train_accuracy':float(clf.score(Xtr,ytr)),'heldout_accuracy':float(clf.score(Xte,yte))},open(os.path.join(WORK_STATE_DIR,'sc_grc_router_config.json'),'w'),indent=2)
    print(f"ROUTER READY | escalation labels={int(y.sum())}/333 | held-out accuracy={clf.score(Xte,yte):.3f}")
else:
    print(f"Shard complete: {START}:{END}. Save WORK_STATE_DIR as a Kaggle Dataset before the next notebook.")


Processing fixed shard 84:168 = 84 questions. This notebook is intended to be run once.


SC-GRC signals 84:168:   0%|          | 0/84 [00:00<?, ?it/s]


initial target device: 100%|██████████| 2/2 [00:19<00:00,  9.51s/it]

Chunks:   0%|          | 0/2 [00:00<?, ?it/s]You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.

Chunks:  50%|█████     | 1/2 [00:02<00:02,  2.51s/it]You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.

Chunks: 100%|██████████| 2/2 [00:03<00:00,  1.53s/it]

Chunks: 100%|██████████| 2/2 [00:00<00:00,  4.94it/s]

Chunks: 100%|██████████| 2/2 [00:00<00:00,  5.15it/s]

Chunks: 100%|██████████| 2/2 [00:00<00:00,  4.84it/s]

Chunks: 100%|██████████| 2/2 [00:00<00:00,  5.56it/s]

Chunks: 100%|██████████| 2/2 [00:00<00:00,  5.37it/s]

Chunks: 100%|██████████| 2/2 [00:00

Shard complete: 84:168. Save WORK_STATE_DIR as a Kaggle Dataset before the next notebook.
